In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pwd

In [ ]:
import os
os.chdir('/content/drive/MyDrive/25-1 티썸')

In [ ]:
os.listdir()

In [ ]:
os.mkdir('/content/seq')

In [ ]:
!unzip -qq "./data/cropped/DFD_original sequences.zip" -d "/content/seq"

In [ ]:
!unzip -qq "./data/cropped/DFD_manipulated_sequences" -d "/content/seq"

In [ ]:
os.rename("/content/seq/DFD_original sequences", "/content/seq/origin")
os.rename("/content/seq/DFD_manipulated_sequences", "/content/seq/manipulated")

In [ ]:
os.listdir('/content/seq/manipulated')

In [ ]:
from google.colab.patches import cv2_imshow
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef
from tqdm import tqdm
import torch.optim as optim
import cv2
import timm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np

In [ ]:
torch.cuda.empty_cache()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
import re
from PIL import Image
from collections import defaultdict
import glob

In [ ]:
class Backbone(nn.Module):
    def __init__(self,
                 model_name,
                 pretrained=True,
                 freeze_until: int = 0):
        """
        freeze_until:
          - 0 이면 아무 것도 얼리지 않음
          - 1 이면 backbone.children()[0] (첫 번째 모듈)까지 얼림
          - 2 이면 children()[:2]까지 얼림, …
        """
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=False,
            num_classes=0
        )
        self.feature_dim = self.backbone.num_features

        # -----------------------
        # 여기서 freeze 제어
        # -----------------------
        if freeze_until > 0:
            children = list(self.backbone.children())
            for child in children[:freeze_until]:
                for param in child.parameters():
                    param.requires_grad = False

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.backbone.global_pool(x)
        return torch.flatten(x, 1)


In [ ]:
class VideoDataset(Dataset):

  def __init__(self, root_dir, frame_num=72, transform=None):
    self.root_dir  = root_dir
    self.frame_num = frame_num
    self.transform = transform
    self.data = []
    self.labels = []

    for label, subdir in enumerate(['origin', 'manipulated']):
      sub_path = os.path.join(root_dir, subdir)
      video_dict = {}

      for img_path in glob.glob(os.path.join(sub_path, "*.jpg")):
        fn = os.path.basename(img_path)
        vid, frame_str, _ = fn.rsplit('_', 2)
        video_dict.setdefault(vid, []).append(img_path)

      for vid, frames in video_dict.items():
        if len(frames) == self.frame_num:
          frames.sort()
          self.data.append({vid: frames})
          self.labels.append({vid: label})

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    # 1) vid와 frame_paths 꺼내기
    vid, frame_paths = next(iter(self.data[idx].items()))
    label = self.labels[idx][vid]          # same vid → label

    # 2) 이미지 로드 및 transform
    imgs = []
    for p in frame_paths:
      img = Image.open(p).convert('RGB')
      if self.transform:
        img = self.transform(img)
      else:
        arr = np.array(img)            # H×W×C, uint8
        img = torch.from_numpy(arr).permute(2,0,1).float().div(255.0)
      imgs.append(img)

    # 3) (T, C, H, W) 형태로 스택
    video = torch.stack(imgs, dim=0)

    # 4) vid 도 함께 반환
    return video, label, vid


In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import torch, glob, os

class DeepfakeVideoDataset(Dataset):
    def __init__(self, root_dir, frame_num=72, size=224, face_count_filter=None):
        """
        root_dir/
            original/
            manipulated/
        face_count_filter:
          None → 모든 비디오
          1    → 얼굴이 정확히 1개인 비디오
          2    → 얼굴이 2개 이상인 비디오
        """
        self.data   = []  # list of list-of-frame-paths
        self.labels = []  # 0=original, 1=manipulated
        self.frame_num = frame_num
        self.size      = size

        # CV2 이미지 → Tensor + Normalize
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],
                                 [0.229,0.224,0.225])
        ])

        for label, subdir in enumerate(['original', 'manipulated']):
            folder = os.path.join(root_dir, subdir)
            video_dict = {}

            # 1) 영상ID → 얼굴인덱스 → 프레임인덱스 → 파일경로
            for img_path in glob.glob(os.path.join(folder, "*.jpg")):
                fname, _ = os.path.splitext(os.path.basename(img_path))
                parts = fname.split('_')
                if len(parts) < 3:
                    continue
                try:
                    frame_idx = int(parts[-2])
                    face_idx  = int(parts[-1])
                except ValueError:
                    continue
                vid = '_'.join(parts[:-2])
                video_dict.setdefault(vid, {}) \
                          .setdefault(face_idx, {})[frame_idx] = img_path

            # 2) 얼굴 개수 필터링 및 완전 시퀀스 수집
            for vid, faces in video_dict.items():
                cnt = len(faces)
                if face_count_filter is not None:
                    if face_count_filter == 1 and cnt != 1:
                        continue
                    if face_count_filter == 2 and cnt < 2:
                        continue

                # 각 얼굴(face_idx)마다 프레임이 frame_num개여야 함
                for frame_dict in faces.values():
                    if len(frame_dict) != self.frame_num:
                        continue
                    frames = [frame_dict[i] for i in range(self.frame_num)]
                    self.data.append(frames)
                    self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        frames = self.data[idx]
        label  = self.labels[idx]
        imgs   = []
        for p in frames:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.size, self.size))
            img = self.transform(img)  # (3, size, size)
            imgs.append(img)
        # (T, C, H, W)
        video_tensor = torch.stack(imgs, dim=0)
        return video_tensor, label

In [ ]:
BATCH_SIZE = 2
LEARNING_RATE = 1e-2
EPOCHS = 10

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
backbone = Backbone('xception', pretrained=True, freeze_until=0).to(device)

In [ ]:
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader, Subset

# # 1) 전체 데이터셋
# # video_data = VideoDataset('/content/seq', transform=transform)
# video_data = DeepfakeVideoDataset('/content/seq')

# # 2) 인덱스 생성
# indices = list(range(len(video_data)))

# # 3) train : (val+test) = 70% : 30% 비율로 분할
# train_idx, temp_idx = train_test_split(
#     indices,
#     test_size=0.3,
#     random_state=42
# )

# # 4) temp = val+test → val : test = 50% : 50% (즉, 전체의 15% : 15%)
# val_idx, test_idx = train_test_split(
#     temp_idx,
#     test_size=0.5,
#     random_state=42
# )

# # 5) Subset 으로 분할
# train_dataset = Subset(video_data, train_idx)
# val_dataset   = Subset(video_data, val_idx)
# test_dataset  = Subset(video_data, test_idx)

# # # 6) DataLoader 생성
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
# val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,  num_workers=4)
# test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)


In [ ]:
ROOT = '/content/seq'    # 실제 경로
ds = DeepfakeVideoDataset(ROOT, frame_num=72, size=224, face_count_filter=1)
print("▶ dataset length:", len(ds))

# 3) 첫 번째 샘플 직접 불러오기
video0, label0 = ds[0]
print("▶ sample[0] video shape:", video0.shape)  # 기대: (72, 3, 224, 224)
print("▶ sample[0] label     :", label0)

In [ ]:
loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)
batch = next(iter(loader))   # 한 배치만 꺼내 봅니다
x, y = batch
print("▶ batch x.shape:", x.shape)  # (2, 72, 3, 224, 224)
print("▶ batch y    :", y)

In [ ]:
import os, glob, cv2, torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split

root_dir   = '/content/seq'
BATCH_SIZE = 4
FRAME_NUM  = 72
IMG_SIZE   = 224

# Train/Val loaders (얼굴 1개)
ds1 = DeepfakeVideoDataset(root_dir, frame_num=FRAME_NUM, size=IMG_SIZE, face_count_filter=1)
labels = [ds1[i][1] for i in range(len(ds1))]
idx    = list(range(len(ds1)))
train_idx, val_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)

train_loader = DataLoader(Subset(ds1, train_idx), batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(Subset(ds1, val_idx),   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Test loader (얼굴 2개 이상)
# ds2 = DeepfakeVideoDataset(root_dir, frame_num=FRAME_NUM, size=IMG_SIZE, face_count_filter=2)
# test_loader = DataLoader(ds2, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
from collections import Counter

# train_dataset, val_dataset, test_dataset는 Subset 객체라고 가정

def get_label_by_idx(idx):
    # video_data.labels[idx] 는 {vid: label} 형태이므로 values()[0]를 씁니다
    return list(video_data.labels[idx].values())[0]

# 1) 각 subset의 실제 인덱스 가져오기
train_idx = train_dataset.indices
val_idx   = val_dataset.indices
test_idx  = test_dataset.indices

# 2) 레이블 리스트 생성
train_labels = [get_label_by_idx(i) for i in train_idx]
val_labels   = [get_label_by_idx(i) for i in val_idx]
test_labels  = [get_label_by_idx(i) for i in test_idx]

# 3) 분포 출력
print("Train label dist:", Counter(train_labels))
print(" Val  label dist:", Counter(val_labels))
print("Test  label dist:", Counter(test_labels))


In [ ]:
# 2) Xception2LSTM 수정: lstm_input_dim을 제거하고 backbone.feature_dim 사용
class Xception2LSTM(nn.Module):
    def __init__(self,
                 backbone: Backbone,
                 lstm_hidden_size: int,
                 lstm_layers: int,
                 num_classes: int):
        super().__init__()
        self.backbone = backbone
        feat_dim = backbone.feature_dim    # 여기서 자동으로 가져옴

        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True
        )
        self.classifier = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape

        # 1) CNN에 프레임별로 통과
        x = x.view(B * T, C, H, W)             # (B*T, C, H, W)
        feats = self.backbone(x)               # (B*T, feat_dim)

        # 2) 다시 시퀀스 차원 복원
        feats = feats.view(B, T, -1)           # (B, T, feat_dim)

        # 3) LSTM
        out, (hn, _) = self.lstm(feats)        # out: (B, T, H), hn: (layers, B, H)

        # 4) 마지막 타임스텝
        last = out[:, -1, :]                   # (B, H)

        # 5) 분류기
        logits = self.classifier(last)         # (B, num_classes)
        return logits


In [ ]:
model = Xception2LSTM(
    backbone=backbone,
    lstm_hidden_size=72,
    lstm_layers=2,
    num_classes=2
).to(device)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=[0.25, 0.75], gamma=2, reduction='mean'):
        """
        alpha: list or tensor of shape [num_classes]
        gamma: focusing parameter
        reduction: 'mean' or 'sum' or 'none'
        """
        super().__init__()
        # 1) alpha를 텐서로 저장
        self.alpha = torch.tensor(alpha, dtype=torch.float)
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        inputs: (B, C) 로짓
        targets: (B,) 정수 레이블
        """
        # 2) 기본 CrossEntropy per-sample
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')  # (B,)
        pt = torch.exp(-ce_loss)                                      # (B,)

        # 3) 각 샘플별 alpha 값 추출
        #    self.alpha (torch.Tensor) 을 GPU에 올리기
        if inputs.device != self.alpha.device:
            self.alpha = self.alpha.to(inputs.device)
        at = self.alpha[targets]                                      # (B,)

        # 4) Focal Loss 계산
        loss = at * (1 - pt) ** self.gamma * ce_loss                  # (B,)

        # 5) Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:  # 'none'
            return loss


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = FocalLoss()

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, matthews_corrcoef
)

In [ ]:
best_auc   = 0.0
patience   = 10
pat        = 0
num_epochs = 10

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc' : [], 'val_acc' : [],
    'train_f1'  : [], 'val_f1'  : [],
    'train_auc' : [], 'val_auc' : [],
    'train_pr_auc': [], 'val_pr_auc': [],
    'train_precision': [], 'val_precision': [],
    'train_recall': [], 'val_recall': [],
    'train_mcc': [], 'val_mcc': []
}

for epoch in range(1, num_epochs+1):
    # --- Train ---
    model.train()
    train_losses, train_preds, train_targets = [], [], []
    for x, y, vid in tqdm(train_loader, desc=f'Epoch {epoch} [Train]'):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        probs = out.softmax(dim=1)[:,1].detach().cpu().tolist()
        train_preds.extend(probs)
        train_targets.extend(y.cpu().tolist())

    # 훈련 지표 계산
    train_loss = np.mean(train_losses)
    train_pred_labels = [1 if p>0.5 else 0 for p in train_preds]
    train_acc       = accuracy_score(train_targets, train_pred_labels)
    train_f1        = f1_score(train_targets, train_pred_labels, zero_division=0)
    train_roc_auc   = roc_auc_score(train_targets, train_preds)
    train_pr_auc    = average_precision_score(train_targets, train_preds)
    train_precision = precision_score(train_targets, train_pred_labels, zero_division=0)
    train_recall    = recall_score(train_targets, train_pred_labels, zero_division=0)
    train_mcc       = matthews_corrcoef(train_targets, train_pred_labels)

    # 기록
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['train_auc'].append(train_roc_auc)
    history['train_pr_auc'].append(train_pr_auc)
    history['train_precision'].append(train_precision)
    history['train_recall'].append(train_recall)
    history['train_mcc'].append(train_mcc)

    # --- Validation ---
    model.eval()
    val_losses, val_preds, val_targets = [], [], []
    with torch.no_grad():
        for x, y, vid in tqdm(val_loader, desc=f'Epoch {epoch} [Val]'):
            x, y = x.to(device), y.to(device)
            out   = model(x)
            loss  = criterion(out, y)

            val_losses.append(loss.item())
            probs = out.softmax(dim=1)[:,1].cpu().tolist()
            val_preds.extend(probs)
            val_targets.extend(y.cpu().tolist())

    # 검증 지표 계산
    val_loss = np.mean(val_losses)
    val_pred_labels = [1 if p>0.5 else 0 for p in val_preds]
    val_acc       = accuracy_score(val_targets, val_pred_labels)
    val_f1        = f1_score(val_targets, val_pred_labels, zero_division=0)
    val_roc_auc   = roc_auc_score(val_targets, val_preds)
    val_pr_auc    = average_precision_score(val_targets, val_preds)
    val_precision = precision_score(val_targets, val_pred_labels, zero_division=0)
    val_recall    = recall_score(val_targets, val_pred_labels, zero_division=0)
    val_mcc       = matthews_corrcoef(val_targets, val_pred_labels)

    # 기록
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['val_auc'].append(val_roc_auc)
    history['val_pr_auc'].append(val_pr_auc)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['val_mcc'].append(val_mcc)

    print(f"[{epoch}/{num_epochs}] "
          f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, val_auc={val_roc_auc:.4f}, "
          f"val_pr_auc={val_pr_auc:.4f}, val_mcc={val_mcc:.4f}")

    # Early Stopping
    if val_roc_auc > best_auc:
        best_auc = val_roc_auc
        torch.save(model.state_dict(), 'best_model.pth')
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

In [ ]:
from collections import Counter

# (A) val_dataset 자체로 레이블 분포 보기
val_labels = [ video_data.labels[i][ next(iter(video_data.data[i])) ]
               for i in val_idx ]
print("Val set dist:", Counter(val_labels))

# (B) val_loader 전체 순회 (shuffle=False 권장)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)
all_y = []
for _, y, _ in val_loader:
    all_y.extend(y.tolist())
print("Val loader dist:", Counter(all_y))


In [ ]:
model.eval()
test_losses, test_preds, test_targets = [], [], []
with torch.no_grad():
    for x, y, vid in tqdm(test_loader, desc='[Test]'):
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)

        test_losses.append(loss.item())
        probs = out.softmax(dim=1)[:, 1].cpu().tolist()
        test_preds.extend(probs)
        test_targets.extend(y.cpu().tolist())

# 손실 및 예측 라벨 계산
test_loss = float(np.mean(test_losses))
test_pred_labels = [1 if p > 0.5 else 0 for p in test_preds]

# 주요 평가지표 계산
test_acc       = accuracy_score(test_targets, test_pred_labels)
test_f1        = f1_score(test_targets, test_pred_labels, zero_division=0)
test_roc_auc   = roc_auc_score(test_targets, test_preds)
test_pr_auc    = average_precision_score(test_targets, test_preds)
test_precision = precision_score(test_targets, test_pred_labels, zero_division=0)
test_recall    = recall_score(test_targets, test_pred_labels, zero_division=0)
test_mcc       = matthews_corrcoef(test_targets, test_pred_labels)

print(f"Test Results ▶ "
      f"loss={test_loss:.4f}, "
      f"acc={test_acc:.4f}, "
      f"f1={test_f1:.4f}, "
      f"pr_auc={test_pr_auc:.4f}, "
      f"roc_auc={test_roc_auc:.4f}, "
      f"precision={test_precision:.4f}, "
      f"recall={test_recall:.4f}, "
      f"mcc={test_mcc:.4f}")


In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.2)

epochs = len(history['train_loss'])
x = np.arange(1, epochs + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
titles = ['Loss', 'Accuracy', 'F1-score', 'AUC']
metrics = [
    ('train_loss', 'val_loss'),
    ('train_acc', 'val_acc'),
    ('train_f1', 'val_f1'),
    ('train_auc', 'val_auc')
]

train_color = '#0066CC'  # 진한 파랑
val_color = '#FF5733'    # 오렌지/빨강

for idx, ax in enumerate(axes.flat):
    tr, va = metrics[idx]
    ax.plot(x, history[tr], label='Train', color=train_color, linewidth=2.2, marker='o', markersize=6)
    ax.plot(x, history[va], label='Val', color=val_color, linewidth=2.2, marker='s', markersize=6)
    ax.set_title(titles[idx], fontsize=17, weight='bold')
    ax.set_xlabel('Epoch', fontsize=13)
    ax.set_ylabel(titles[idx], fontsize=13)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=12, loc='best')
    # Best epoch marker for val
    best_idx = np.argmax(history[va]) if 'loss' not in va else np.argmin(history[va])
    ax.scatter(x[best_idx], history[va][best_idx],
               color=val_color, s=120, edgecolor='black',
               label='Best Val', zorder=5)
    if idx == 0:  # Loss만 ymin 조정
        ax.set_ylim(bottom=0)

plt.suptitle("Training vs. Validation Metrics per Epoch", fontsize=22, weight='bold', color='#1a1a1a', y=1.03)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()